# Data & Point-in-Time Universe

## Crypto Alpha Across Horizons

This notebook documents and validates the market-data foundation used by the research project before any signal, portfolio, or performance analysis.

### Research data design

- **Venue:** Binance global spot
- **Quote currency:** USDT
- **Universe:** monthly top 25 eligible assets by trailing liquidity
- **Minimum listing history:** 90 days
- **Liquidity measure:** trailing 30-day median daily quote volume
- **Universe refresh:** start of each UTC calendar month
- **Eligibility:** ordinary crypto spot risk assets; stable/fiat-like assets, obvious wrapped or pegged duplicates, commodity-pegged assets, leveraged tokens, and non-standard products are excluded before ranking
- **Missing data:** observations remain missing; prices and returns are never silently forward-filled

The initial research cycle used Binance-native **4-hour bars**. After the original strategy failed validation, the later continuation redesign used separately acquired Binance-native **1-hour bars** while retaining the same point-in-time universe discipline. That later 1-hour acquisition is introduced with the redesign rather than retroactively changing this initial data layer.

> **Reproducibility note.** The original broad candidate pool came from Binance `exchangeInfo`, which is live exchange metadata and changes over time. During the pre-signal data-integrity stage, the intended asset-type exclusions were corrected and the resulting historical universe was frozen. Public reproduction should therefore use the frozen canonical membership artifact rather than rebuild historical membership from today's exchange metadata.

> **Residual limitation.** Because the original candidate pool was based on exchange metadata available at the time of acquisition, assets delisted before that pull may be absent. This residual survivorship limitation is preserved explicitly.

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def locate_project_root():
    """Locate the local project root from common notebook launch locations."""
    env_root = os.getenv("CRYPTO_ALPHA_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "data").is_dir():
            return candidate

    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "crypto-alpha-research",
        cwd.parent / "crypto-alpha-research",
    ]

    for candidate in candidates:
        if (candidate / "data").is_dir() and (
            (candidate / "notebooks").is_dir()
            or (candidate / "public_notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Set the CRYPTO_ALPHA_ROOT "
        "environment variable to the crypto-alpha-research folder."
    )


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_4H_DIR = DATA_DIR / "processed"
UNIVERSE_DIR = DATA_DIR / "universe"

START_DATE = "2020-01-01"
END_DATE = "2026-08-01"
RESEARCH_END_EXCLUSIVE = pd.Timestamp(END_DATE, tz="UTC")

UNIVERSE_SIZE = 25
MIN_HISTORY_DAYS = 90
LIQUIDITY_WINDOW_DAYS = 30
EXPECTED_4H_FREQ = pd.Timedelta(hours=4)

print("Project root:", PROJECT_ROOT)
print("Universe directory:", UNIVERSE_DIR)
print("4h processed-data directory:", PROCESSED_4H_DIR)

Project root: /Users/owensimon/crypto-alpha-research
Universe directory: /Users/owensimon/crypto-alpha-research/data/universe
4h processed-data directory: /Users/owensimon/crypto-alpha-research/data/processed


## 1. Market-data acquisition

Binance klines are retrieved directly from the public spot API. The historical downloader uses pagination and treats requested intervals as **start-inclusive and end-exclusive**. Raw API responses were preserved locally during the original build, while cleaned research tables were stored as Parquet.

The functions below retain the core acquisition and cleaning logic without re-running the expensive full download when this notebook is executed.

In [2]:
KLINES_URL = "https://data-api.binance.vision/api/v3/klines"


def download_raw_klines(symbol, start_date, end_date, interval="4h", limit=1000):
    """Download Binance spot klines over a historical UTC date range."""
    start_ms = int(pd.Timestamp(start_date, tz="UTC").timestamp() * 1000)
    end_ms = int(pd.Timestamp(end_date, tz="UTC").timestamp() * 1000) - 1

    rows = []
    current_start = start_ms

    while current_start < end_ms:
        response = requests.get(
            KLINES_URL,
            params={
                "symbol": symbol,
                "interval": interval,
                "startTime": current_start,
                "endTime": end_ms,
                "limit": limit,
            },
            timeout=30,
        )
        response.raise_for_status()
        batch = response.json()

        if not batch:
            break

        rows.extend(batch)
        current_start = batch[-1][0] + 1

        if len(batch) < limit:
            break

    return rows


def clean_klines(raw_data):
    """Convert raw Binance kline rows into the standardized research schema."""
    columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "base_volume",
        "close_time",
        "quote_volume",
        "num_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore",
    ]

    df = pd.DataFrame(raw_data, columns=columns)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

    float_columns = [
        "open",
        "high",
        "low",
        "close",
        "base_volume",
        "quote_volume",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
    ]
    df[float_columns] = df[float_columns].astype(float)
    df["num_trades"] = df["num_trades"].astype(int)

    return df.drop(columns="ignore")

## 2. Point-in-time monthly universe

At each monthly decision boundary, universe membership is determined only from information available before that month begins:

1. require at least **90 days** of observed history;
2. calculate each asset's trailing **30-day median daily quote volume**;
3. rank eligible assets cross-sectionally;
4. select the **top 25**.

The function below is the core historical ranking rule. It does **not** query current `exchangeInfo`; the frozen corrected membership is loaded in the next section so that mutable live metadata cannot silently redefine the experiment.

In [3]:
def build_monthly_universe(
    daily_data,
    top_n=UNIVERSE_SIZE,
    min_history_days=MIN_HISTORY_DAYS,
    liquidity_window=LIQUIDITY_WINDOW_DAYS,
):
    """Construct point-in-time monthly liquidity-ranked membership."""
    records = []

    for symbol, df in daily_data.items():
        x = df[["open_time", "quote_volume"]].sort_values("open_time").copy()
        x["date"] = x["open_time"].dt.floor("D")
        first_date = x["date"].min()

        x["liquidity_30d_median"] = (
            x["quote_volume"]
            .rolling(window=liquidity_window, min_periods=liquidity_window)
            .median()
        )

        # Information through the completed prior month determines
        # membership for the next UTC calendar month.
        x["effective_month"] = x["date"] + pd.offsets.MonthBegin(1)

        monthly_info = x.groupby("effective_month", as_index=False).last()
        monthly_info["history_days"] = (
            monthly_info["date"] - first_date
        ).dt.days + 1
        monthly_info["symbol"] = symbol

        records.append(
            monthly_info[
                [
                    "effective_month",
                    "symbol",
                    "history_days",
                    "liquidity_30d_median",
                ]
            ]
        )

    universe_panel = pd.concat(records, ignore_index=True)
    eligible_panel = universe_panel.loc[
        (universe_panel["history_days"] >= min_history_days)
        & universe_panel["liquidity_30d_median"].notna()
    ].copy()

    eligible_panel["liquidity_rank"] = (
        eligible_panel.groupby("effective_month")["liquidity_30d_median"]
        .rank(method="first", ascending=False)
    )

    membership = (
        eligible_panel.loc[eligible_panel["liquidity_rank"] <= top_n]
        .sort_values(["effective_month", "liquidity_rank"])
        .reset_index(drop=True)
    )

    return universe_panel, membership

## 3. Frozen canonical universe

The initial integrity review found that the intended product-type eligibility rule needed to be enforced **before** monthly liquidity ranking. That repair occurred before any signal, IC, backtest, validation, or final-test performance was examined. Affected months were rebuilt from contemporaneously available liquidity information, the next-highest eligible assets were admitted, and the corrected universe was frozen.

The public research input is the resulting **research-only monthly membership**: 76 usable months from April 2020 through July 2026. It is loaded rather than rebuilt from live metadata. The original local build also retained a full 77-effective-month artifact through 2026-08-01, but that final effective month lies outside the research-data boundary and is not needed for the public research sequence.

In [4]:
def load_research_universe():
    """Load the frozen research membership, preferring a GitHub-friendly CSV."""
    csv_path = UNIVERSE_DIR / "research_monthly_universe.csv"
    parquet_path = UNIVERSE_DIR / "research_monthly_universe.parquet"

    if csv_path.exists():
        frame = pd.read_csv(csv_path)
    elif parquet_path.exists():
        frame = pd.read_parquet(parquet_path)
    else:
        raise FileNotFoundError(
            "Missing frozen research universe. Expected "
            "research_monthly_universe.csv or .parquet in data/universe/."
        )

    frame["effective_month"] = pd.to_datetime(frame["effective_month"], utc=True)
    return frame


research_universe = load_research_universe()
research_selected_symbols = sorted(research_universe["symbol"].unique())

month_counts = research_universe.groupby("effective_month")["symbol"].nunique()
duplicate_memberships = int(
    research_universe[["effective_month", "symbol"]].duplicated().sum()
)
out_of_boundary_rows = int(
    (research_universe["effective_month"] >= RESEARCH_END_EXCLUSIVE).sum()
)

universe_summary = pd.DataFrame(
    {
        "metric": [
            "Usable research months",
            "First research month",
            "Last research month",
            "Assets per month (min)",
            "Assets per month (max)",
            "Research symbol union",
            "Duplicate memberships",
            "Membership rows at/after research end",
        ],
        "value": [
            research_universe["effective_month"].nunique(),
            research_universe["effective_month"].min(),
            research_universe["effective_month"].max(),
            int(month_counts.min()),
            int(month_counts.max()),
            len(research_selected_symbols),
            duplicate_memberships,
            out_of_boundary_rows,
        ],
    }
)

display(universe_summary)

assert research_universe["effective_month"].nunique() == 76
assert research_universe["effective_month"].min() == pd.Timestamp("2020-04-01", tz="UTC")
assert research_universe["effective_month"].max() == pd.Timestamp("2026-07-01", tz="UTC")
assert month_counts.min() == month_counts.max() == 25
assert len(research_selected_symbols) == 151
assert duplicate_memberships == 0
assert out_of_boundary_rows == 0

print("Frozen-universe structure: PASS")

,metric,value
0,Usable research months,76
1,First research month,2020-04-01 00:00:00+00:00
2,Last research month,2026-07-01 00:00:00+00:00
3,Assets per month (min),25
4,Assets per month (max),25
5,Research symbol union,151
6,Duplicate memberships,0
7,Membership rows at/after research end,0


Frozen-universe structure: PASS


## 4. Structural integrity of the frozen 4-hour research layer

The final audit checks the exact corrected universe consumed by the initial research cycle. It verifies:

- required schema and dtypes;
- chronological order;
- duplicate and off-grid timestamps;
- nulls and basic OHLC/volume validity;
- active-universe coverage month by month.

Missing observations remain missing. The project uses exact elapsed-time endpoints later, so a signal or return is calculated only when the required observations exist.

Some historical Binance candles closed earlier than the nominal four-hour endpoint while remaining otherwise valid. Those candles are retained, but their information is treated as available **no earlier than the nominal 4-hour boundary**.

In [5]:
EXPECTED_COLUMNS = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "close_time",
    "quote_volume",
    "num_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
]

FLOAT_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "quote_volume",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
]

file_map = {
    path.name.removesuffix("_4h.parquet"): path
    for path in PROCESSED_4H_DIR.glob("*_4h.parquet")
}

asset_records = []

for symbol in sorted(research_selected_symbols):
    path = file_map.get(symbol)

    if path is None:
        asset_records.append(
            {
                "symbol": symbol,
                "file_exists": False,
                "missing_columns": EXPECTED_COLUMNS.copy(),
                "dtype_issues": ["file_missing"],
                "chronological": False,
                "duplicate_open_times": np.nan,
                "misaligned_open_times": np.nan,
                "total_nulls": np.nan,
                "price_nonpositive": np.nan,
                "negative_volume": np.nan,
                "bad_ohlc": np.nan,
                "shortened_candles": np.nan,
            }
        )
        continue

    df = pd.read_parquet(path)
    missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
    dtype_issues = []

    if "open_time" in df and not isinstance(df["open_time"].dtype, pd.DatetimeTZDtype):
        dtype_issues.append("open_time_not_timezone_aware")
    if "close_time" in df and not isinstance(df["close_time"].dtype, pd.DatetimeTZDtype):
        dtype_issues.append("close_time_not_timezone_aware")

    for col in FLOAT_COLUMNS:
        if col in df and not pd.api.types.is_float_dtype(df[col]):
            dtype_issues.append(f"{col}_not_float")

    if "num_trades" in df and not pd.api.types.is_integer_dtype(df["num_trades"]):
        dtype_issues.append("num_trades_not_integer")

    if missing_columns:
        asset_records.append(
            {
                "symbol": symbol,
                "file_exists": True,
                "missing_columns": missing_columns,
                "dtype_issues": dtype_issues,
                "chronological": False,
                "duplicate_open_times": np.nan,
                "misaligned_open_times": np.nan,
                "total_nulls": np.nan,
                "price_nonpositive": np.nan,
                "negative_volume": np.nan,
                "bad_ohlc": np.nan,
                "shortened_candles": np.nan,
            }
        )
        continue

    chronological_original = bool(df["open_time"].is_monotonic_increasing)
    x = df.sort_values("open_time").reset_index(drop=True)

    aligned = (
        x["open_time"].dt.minute.eq(0)
        & x["open_time"].dt.second.eq(0)
        & x["open_time"].dt.microsecond.eq(0)
        & x["open_time"].dt.hour.mod(4).eq(0)
    )

    bad_ohlc = (
        (x["high"] < x[["open", "close", "low"]].max(axis=1))
        | (x["low"] > x[["open", "close", "high"]].min(axis=1))
    )

    nominal_close = x["open_time"] + EXPECTED_4H_FREQ - pd.Timedelta(milliseconds=1)

    asset_records.append(
        {
            "symbol": symbol,
            "file_exists": True,
            "missing_columns": missing_columns,
            "dtype_issues": dtype_issues,
            "chronological": chronological_original,
            "duplicate_open_times": int(x["open_time"].duplicated().sum()),
            "misaligned_open_times": int((~aligned).sum()),
            "total_nulls": int(x[EXPECTED_COLUMNS].isna().sum().sum()),
            "price_nonpositive": int(
                (
                    (x["open"] <= 0)
                    | (x["high"] <= 0)
                    | (x["low"] <= 0)
                    | (x["close"] <= 0)
                ).sum()
            ),
            "negative_volume": int(
                (
                    (x["base_volume"] < 0)
                    | (x["quote_volume"] < 0)
                    | (x["taker_buy_base_volume"] < 0)
                    | (x["taker_buy_quote_volume"] < 0)
                ).sum()
            ),
            "bad_ohlc": int(bad_ohlc.sum()),
            "shortened_candles": int((x["close_time"] < nominal_close).sum()),
        }
    )

asset_audit = pd.DataFrame(asset_records)

coverage_records = []

for membership in (
    research_universe[["effective_month", "symbol"]]
    .drop_duplicates()
    .itertuples(index=False)
):
    month_start = membership.effective_month
    month_end = min(
        month_start + pd.offsets.MonthBegin(1),
        RESEARCH_END_EXCLUSIVE,
    )

    expected = pd.date_range(
        start=month_start,
        end=month_end - EXPECTED_4H_FREQ,
        freq=EXPECTED_4H_FREQ,
    )

    path = file_map.get(membership.symbol)
    if path is None:
        observed_count = 0
    else:
        opens = pd.read_parquet(path, columns=["open_time"])["open_time"]
        observed = pd.DatetimeIndex(
            opens.loc[(opens >= month_start) & (opens < month_end)]
            .drop_duplicates()
        )
        observed_count = len(expected) - len(expected.difference(observed))

    coverage_records.append(
        {
            "effective_month": month_start,
            "symbol": membership.symbol,
            "expected_bars": len(expected),
            "observed_bars": observed_count,
            "missing_bars": len(expected) - observed_count,
            "coverage": observed_count / len(expected) if len(expected) else np.nan,
        }
    )

monthly_coverage = pd.DataFrame(coverage_records)

integrity_summary = pd.DataFrame(
    {
        "check": [
            "Missing research-universe files",
            "Assets with missing schema columns",
            "Assets with dtype issues",
            "Assets not chronologically ordered",
            "Assets with duplicate opens",
            "Assets with misaligned opens",
            "Assets with nulls",
            "Assets with nonpositive prices",
            "Assets with negative volume",
            "Assets with OHLC inconsistencies",
            "Assets containing shortened valid candles",
        ],
        "count": [
            int((~asset_audit["file_exists"]).sum()),
            int(asset_audit["missing_columns"].map(len).gt(0).sum()),
            int(asset_audit["dtype_issues"].map(len).gt(0).sum()),
            int((~asset_audit["chronological"]).sum()),
            int(asset_audit["duplicate_open_times"].fillna(0).gt(0).sum()),
            int(asset_audit["misaligned_open_times"].fillna(0).gt(0).sum()),
            int(asset_audit["total_nulls"].fillna(0).gt(0).sum()),
            int(asset_audit["price_nonpositive"].fillna(0).gt(0).sum()),
            int(asset_audit["negative_volume"].fillna(0).gt(0).sum()),
            int(asset_audit["bad_ohlc"].fillna(0).gt(0).sum()),
            int(asset_audit["shortened_candles"].fillna(0).gt(0).sum()),
        ],
    }
)

incomplete_memberships = monthly_coverage.loc[
    monthly_coverage["missing_bars"] > 0
].sort_values(["effective_month", "symbol"])

display(integrity_summary)

coverage_summary = pd.DataFrame(
    {
        "metric": [
            "Membership-month observations",
            "Mean membership-month coverage",
            "Minimum membership-month coverage",
            "Incomplete membership-months",
        ],
        "value": [
            len(monthly_coverage),
            monthly_coverage["coverage"].mean(),
            monthly_coverage["coverage"].min(),
            len(incomplete_memberships),
        ],
    }
)
display(coverage_summary)

print("Incomplete membership-months:")
display(incomplete_memberships)

hard_issue_checks = integrity_summary.iloc[:10]
assert hard_issue_checks["count"].eq(0).all()
assert len(incomplete_memberships) == 1

luna_gap = incomplete_memberships.iloc[0]
assert luna_gap["symbol"] == "LUNAUSDT"
assert luna_gap["effective_month"] == pd.Timestamp("2022-05-01", tz="UTC")
assert int(luna_gap["expected_bars"]) == 186
assert int(luna_gap["observed_bars"]) == 78
assert int(luna_gap["missing_bars"]) == 108

print("Frozen 4h structural-integrity gate: PASS")

,check,count
0,Missing research-universe files,0
1,Assets with missing schema columns,0
2,Assets with dtype issues,0
3,Assets not chronologically ordered,0
4,Assets with duplicate opens,0
5,Assets with misaligned opens,0
6,Assets with nulls,0
7,Assets with nonpositive prices,0
8,Assets with negative volume,0
9,Assets with OHLC inconsistencies,0


,metric,value
0,Membership-month observations,1900.000000
1,Mean membership-month coverage,0.999694
2,Minimum membership-month coverage,0.419355
3,Incomplete membership-months,1.000000


Incomplete membership-months:


,effective_month,symbol,expected_bars,observed_bars,missing_bars,coverage
627,2022-05-01 00:00:00+00:00,LUNAUSDT,186,78,108,0.419355


Frozen 4h structural-integrity gate: PASS


## 5. Data-quality conclusion

The frozen research universe contains **76 usable months (April 2020 through July 2026), 25 assets per month, and 151 unique research assets**. The corrected full universe contains 77 effective months and a 153-symbol union; the unused 2026-08-01 membership lies outside the data boundary.

For the native 4-hour research layer:

- every research-universe asset has a processed file;
- schema, dtype, chronology, duplicate, timestamp-alignment, null, price, volume, and OHLC checks pass;
- the only incomplete active membership is **LUNAUSDT in May 2022**, with 78 of 186 expected 4-hour bars observed and 108 left missing;
- shortened but otherwise valid Binance candles are retained and treated as available only at the nominal 4-hour endpoint.

No signal, information coefficient, backtest, Sharpe ratio, validation result, or final-holdout result is computed in this notebook. The next notebook begins the training-sample signal research from this frozen point-in-time universe.

### Data limitations carried forward

1. The historical candidate pool inherits a residual survivorship limitation from the live Binance metadata available during the original acquisition.
2. Spot data are used later to study theoretical long-short portfolios; real shorting, borrow, futures funding, margin, and venue constraints are not represented by this data layer.
3. Native 1-hour data used in the later continuation redesign are a separate acquisition layer introduced after the original validation failure; they do not retroactively alter the initial 4-hour research record.